Do I have enough data for later layers actually?  
As an example, the last layer has a lot of dimensions. We have only 32 images, 32 true images which induce the pattern we want.  
The problem with taking a lot more images is that the amount of work to do increases manifold.  

The problem is originally from thresholding itself, im not sure what is the right way for this.  
For every neuron, we take a huge set to test from, the majority that we pick is noise.  


Currently, the biggest problem is that of not having a global attribution threshold.  
It would be useful to actually see what the attributions of extracted patches is right now, cuz that is one unknown after correct data normalization that i've not checked out.  


In [ ]:
%config InteractiveShell.cache_size = 0
%load_ext autoreload
%autoreload 2

In [ ]:
from lucent.modelzoo import inceptionv1
import lucent
import matplotlib.pyplot as plt
from lucent.optvis import render, param, transform, objectives
from lucent.modelzoo import inceptionv1
from pathlib import Path
import torch
from lucent.optvis.objectives import wrap_objective, handle_batch
from torch.nn import functional as F
import numpy as np
from olt.tfms import transform, inverse_transform
from PIL import Image
from olt.act import InputOutputModelSnapshot
import torch
from olt.html_report import apply_cmap, to_pil, rd_bk_gn
from olt.shards import read_image_shard
import warnings
from tqdm import tqdm
from olt.feature_viz import tensor_to_img_array
from olt.feature_viz import get_feature_viz_input


device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


imagenet_label = 159

plt.style.use("dark_background")

In [ ]:
keys, images = next(read_image_shard(f"imagenet-label-{imagenet_label}-000000.tar", 128, {}))

In [ ]:
model

In [ ]:
# since inception does GAP, it really just has a single linear layer after it finishes with convolutions lol
# we first extract the PWs of the last layer.

In [ ]:
layer_name = "softmax2_pre_activation_matmul"

In [ ]:
acts = InputOutputModelSnapshot.get_activations(batch, model, [layer_name])
layer_weight = model.get_submodule(layer_name).weight[imagenet_label].detach()
pws = layer_weight * acts[layer_name]["input"]

In [ ]:
32*32

In [ ]:
from olt.show import show_single_channel_red_green_black as S

In [ ]:
# wow tis an extremely dense layer, no kidding. the 0s are very few
# im guessing the input itself is then quite simpler
S([layer_weight.reshape(32,32)], mode="dark")

In [ ]:
from sklearn.preprocessing import normalize

In [ ]:
# i dont expect the inputs coming inf rom the other side to be independent technically, lets still try ica

In [ ]:
# there are 32 of them, we have 8*4 32, so 8 cols, 4 rows
S([normalize(p.reshape(32,32), "l2") for p in pws], (8*3,4*3), 8, mode="dark")
plt.show()

In [ ]:
from sklearn.decomposition import MiniBatchDictionaryLearning, FastICA, SparsePCA

In [ ]:
pws.shape

In [ ]:
dl = MiniBatchDictionaryLearning(16)
dl.fit(pws.numpy())

In [ ]:
codes = dl.transform(pws)

In [ ]:
recon = codes @ dl.components_

In [ ]:
recon[0].shape, pws[0].shape

In [ ]:
S([recon[0].reshape(32,32), pws[0].reshape(32,32)], mode="dark")
plt.show()

In [ ]:
codes[0]

In [ ]:
codes[0]

In [ ]:
plt.plot(codes[0])

In [ ]:
dl.components_.shape

In [ ]:
ica = FastICA(4, whiten="arbitrary-variance", max_iter=10_000)
ica.fit(pws)

ica.mixing_.shape

In [ ]:
S([c.reshape(32,32) for c in ica.mixing_.T], 20, 4)

In [ ]:
from captum.attr import DeepLift

In [ ]:
dlft = DeepLift(model)


In [ ]:
inv_img.shape

In [ ]:
batch.shape

In [ ]:

def make_all_heatmaps():
    cells = []
    for i in range(len(batch)):
        size = (224, 224)
        inv_img = inverse_transform(batch[i]).permute(1,2,0)
        base = to_pil(inv_img, size=size)
        attr_res = dlft.attribute(batch[i][None], target=imagenet_label)
        overlay = attr_res[0].detach().cpu().sum(dim=0).numpy()
        overlay = overlay / np.abs(overlay).max()
        
        # print(base.shape, overlay.shape)
        
        
        heat = apply_cmap(
            overlay,
            rd_bk_gn,
            vmin=-1,
            vmax=1,
            size=size,
            interpolation=Image.BILINEAR,
        )
        cell = Image.blend(base, heat, alpha=0.8)
        cells.append(cell)
        
    return cells

In [ ]:
images = make_all_heatmaps()

In [ ]:
images

In [ ]:
import matplotlib.pyplot as plt

def show_grid_matplotlib(image_list, rows, cols):
    # Initialize the figure layout
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    
    # Flatten axes array for easy 1D iteration
    axes = axes.flatten()
    
    for i, img in enumerate(image_list):
        if i < len(axes):
            axes[i].imshow(img)
            axes[i].axis('off')  # Hide the X/Y coordinate ticks
            
    # Hide any remaining empty subplots if image_list is shorter than rows * cols
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    

It seems simple enough i think, the same kind of integrated gradients gives the same kinda patterns. I should be able to cluster this. Lets try that first.  I'm not sure about curse of dimensionality but lets see. We use kmeans cuz we assume that there is no noise.  

Note that it is known that kmeans works well on the last layer anyways, i think its a good signal here. We will also then try to see the maximally activating things deeplift.   

In [ ]:
S([pws[9].reshape(32,32), pws[18].reshape(32,32), pws[1].reshape(32,32), pws[26].reshape(32,32)], 20, 4)

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
cluster = KMeans(10).fit(normalize(pws, "l2"))

In [ ]:
from collections import defaultdict
idx_by_images = defaultdict(list)

for i, l in enumerate(cluster.labels_):
    idx_by_images[l].append(images[i])

In [ ]:
# elbows not coming hmmm
inertias = []
k_range = range(1, 32, 4)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(pws)
    inertias.append(km.inertia_)

plt.plot(k_range, inertias, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

In [ ]:
for k in idx_by_images:
    print("#####################", k, "#######################")
    show_grid_matplotlib(idx_by_images[k], 1, 10)
    plt.show()

In [ ]:
# from hdbscan import HDBSCAN
from sklearn.cluster import HDBSCAN

In [ ]:
cluster = HDBSCAN(min_cluster_size=2, cluster_selection_method="leaf")
cluster.fit(normalize(pws, "l2"))

In [ ]:
from collections import defaultdict
idx_by_images = defaultdict(list)

for i, l in enumerate(cluster.labels_):
    idx_by_images[l].append(images[i])

In [ ]:
for k in idx_by_images:
    if k == -1:
        continue
    print("#####################", k, "#######################")
    show_grid_matplotlib(idx_by_images[k], 1, 10)
    plt.show()

In [ ]:
centered_pws = pws - pws.mean(dim=0)

In [ ]:
pca = SparsePCA(5).fit(centered_pws)

In [ ]:
S([c.reshape(32,32) for c in pca.components_], 20, 5)

# We need more data

Not enough data for any analysis right now. lets do tabby cat for which we have quite a lot of data. 


Clustering btw does not seem like its working here, sadly. im not sure why though ;_;   

HDBSCAN just marks everything as noise lol, is the curse of dimensionality? It might be useful to try dict learning then.  

In [ ]:
! aws s3 sync s3://narang99-private/imagenet/281/ ./tabby_cat

In [ ]:
! ls tabby_cat | wc -l

In [ ]:
# good, inceptionv1 uses older label files, in that tabby cat is 174, we'll use this as target neuron

In [ ]:
! ls tabby_cat | head

In [ ]:
imagenet_label = 174

In [ ]:
pils = [Image.open(p) for p in Path("tabby_cat").glob("*.jpg")]

In [ ]:
final_pils = []
timgs = []

for p in pils:
    timg = transform(p)
    if timg.shape[0] == 3:
        final_pils.append(p)
        timgs.append(timg)

In [ ]:
len(final_pils), len(timgs)

In [ ]:
from collections import Counter
Counter(tuple(timg.shape) for timg in timgs)

In [ ]:
layer_weight = model.get_submodule(layer_name).weight[imagenet_label].detach()
all_pws = []

indexes = []
for i in tqdm(range(0, len(timgs), 8)):
    batch  = torch.stack(timgs[i:i+8])
    acts = InputOutputModelSnapshot.get_activations(batch, model, [layer_name])
    pws = layer_weight * acts[layer_name]["input"]
    all_pws.append(pws)

In [ ]:
all_pws = torch.cat(all_pws).numpy()

In [ ]:
import gc
gc.collect()

In [ ]:
from sklearn.decomposition import MiniBatchDictionaryLearning, SparsePCA

In [ ]:
def _get_sorted_comps(code, comps):
    new_comps = code[..., None] * comps
    idxs = np.argsort(new_comps.mean(axis=1))[::-1]
    return new_comps[idxs]

In [ ]:
dl = MiniBatchDictionaryLearning(300)

In [ ]:
dl.fit(all_pws)

In [ ]:
(recon - all_pws).sum()

In [ ]:
top_comps = _get_sorted_comps(codes[0], dl.components_)[:5]
S([t.reshape(32,32) for t in top_comps], (20,7), 5)
plt.show()

In [ ]:
from olt.show import show_single_channel_red_green_black as S


S([recon[0].reshape(32,32), all_pws[0].reshape(32,32)])

In [ ]:
recon = codes @ dl.components_

In [ ]:
pca = SparsePCA(100, alpha=0.1)
pca.fit(all_pws)

In [ ]:
codes= pca.transform(all_pws)

In [ ]:
recon = codes @ pca.components_

In [ ]:
from olt.show import show_single_channel_red_green_black as S


S([recon[0].reshape(32,32), all_pws[0].reshape(32,32)])

In [ ]:
plt.plot(codes[0])

In [ ]:
top_comps = _get_sorted_comps(codes[0], pca.components_)[:5]
S([t.reshape(32,32) for t in top_comps], (20,7), 5)
plt.show()

In [ ]:
all_pws.shape

In [ ]:
# we would like to see the clusters until 11 lets say, with minimum difference
# we ll check the outliers later.


km = KMeans(15)
km.fit(all_pws)

In [ ]:

closest = {}
for cluster_id in range(len(km.cluster_centers_)):
    mask = np.where(km.labels_ == cluster_id)[0]
    center = km.cluster_centers_[cluster_id]
    distances = np.linalg.norm(all_pws[mask] - center, axis=1)
    closest[cluster_id] = mask[np.argsort(distances)[:10]]

In [ ]:
def make_heatmaps(timgs, imagenet_label):
    cells = []
    for i in tqdm(range(len(timgs))):
        size = (224, 224)
        inv_img = inverse_transform(timgs[i]).permute(1,2,0)
        base = to_pil(inv_img, size=size)
        attr_res = dlft.attribute(timgs[i][None], target=imagenet_label)
        overlay = attr_res[0].detach().cpu().sum(dim=0).numpy()
        overlay = overlay / np.abs(overlay).max()
        
        # print(base.shape, overlay.shape)
        
        
        heat = apply_cmap(
            overlay,
            rd_bk_gn,
            vmin=-1,
            vmax=1,
            size=size,
            interpolation=Image.BILINEAR,
        )
        cell = Image.blend(base, heat, alpha=0.8)
        cells.append(cell)
    return cells

In [ ]:
heatmaps = make_heatmaps(timgs, imagenet_label)

In [ ]:
for cluster_id, closest_idxs in closest.items():
    hms = [heatmaps[idx] for idx in closest_idxs]
    # hms = [final_pils[idx] for idx in closest_idxs]
    print("#####################", cluster_id, "#######################")
    show_grid_matplotlib(hms, 1, 10)
    plt.show()

In [ ]:
# import numpy as np

# km = KMeans(n_clusters=1, random_state=42, n_init='auto')
# km.fit(pws)

# center = km.cluster_centers_[0]
# distances = np.linalg.norm(pws - center, axis=1)
# closest_indices = np.argsort(distances)[:10]

In [ ]:
cluster = HDBSCAN(min_cluster_size=5, cluster_selection_method="leaf")
cluster.fit(normalize(all_pws, "l2"))

In [ ]:
np.unique(cluster.labels_, return_counts=True)

### NMF


In [ ]:
layer_weight = model.get_submodule(layer_name).weight[imagenet_label].detach()
all_patches = []

indexes = []
for i in tqdm(range(0, len(timgs), 8)):
    batch  = torch.stack(timgs[i:i+8])
    acts = InputOutputModelSnapshot.get_activations(batch, model, [layer_name])
    patch = acts[layer_name]["input"]
    all_patches.append(patch)

In [ ]:
gc.collect()

In [ ]:
from sklearn.decomposition import NMF

In [ ]:
all_patches = torch.cat(all_patches).numpy()

In [ ]:
nmf = NMF(20, max_iter=1000)


In [ ]:
nmf = nmf.fit(all_patches)


In [ ]:
codes = nmf.transform(all_patches)

In [ ]:
# good, sparse enough i guess
codes[0]

In [ ]:
S([c.reshape(32,32) for c in nmf.components_], (4*3,5*3), 5)

In [ ]:
main_comps = []
for i in range(len(codes[0])):
    if codes[0][i] < 0.02:
        continue
    main_comps.append(codes[0][i] * nmf.components_[i])
# scaled = codes[0][..., None] * nmf.components_

In [ ]:
len(main_comps)

In [ ]:
S([all_patches[0].reshape(32,32)], (3,3))

In [ ]:
S([c.reshape(32,32) for c in main_comps], (20,8), 5)

In [ ]:
# hard to guess how good this decomposition is.   
# one thing i can try is do feature visualisation on this vector

In [ ]:
@wrap_objective()
def exact_tensor_all_chans(layer, value, batch=None):
    @handle_batch(batch)
    def inner(model):
        o = model(layer)
        # print(value.shape)
        res = F.mse_loss(o, value)
        return res
    return inner


In [ ]:
all_patches[0].shape

In [ ]:

import warnings
from collections import OrderedDict
import numpy as np
from tqdm import tqdm
from PIL import Image
import torch

from lucent.optvis import objectives, transform, param
from lucent.misc.io import show


class ModuleHook:
    def __init__(self, module):
        self.hook = module.register_forward_hook(self.hook_fn)
        self.module = None
        self.features = None

    def hook_fn(self, module, input, output):
        self.module = module
        self.features = input[0]

    def close(self):
        # This doesn't actually do anything
        self.hook.remove()


def hook_model(model, image_f, return_hooks=False):
    features = OrderedDict()

    # recursive hooking function
    def hook_layers(net, prefix=[]):
        if hasattr(net, "_modules"):
            for name, layer in net._modules.items():
                if layer is None:
                    # e.g. GoogLeNet's aux1 and aux2 layers
                    continue
                features["_".join(prefix + [name])] = ModuleHook(layer)
                hook_layers(layer, prefix=prefix + [name])

    hook_layers(model)

    def hook(layer):
        if layer == "input":
            out = image_f()
        elif layer == "labels":
            out = list(features.values())[-1].features
        else:
            assert layer in features, f"Invalid layer {layer}. Retrieve the list of layers with `lucent.modelzoo.util.get_model_layers(model)`."
            out = features[layer].features
        assert out is not None, "There are no saved feature maps. Make sure to put the model in eval mode, like so: `model.to(device).eval()`. See README for example."
        return out

    if return_hooks:
        return hook, features
    return hook



def render_vis(
    model,
    objective_f,
    param_f=None,
    optimizer=None,
    transforms=None,
    thresholds=(512,),
    verbose=False,
    preprocess=True,
    progress=True,
    show_image=True,
    save_image=False,
    image_name=None,
    show_inline=False,
    fixed_image_size=None,
):
    if param_f is None:
        param_f = lambda: param.image(128)
    # param_f is a function that should return two things
    # params - parameters to update, which we pass to the optimizer
    # image_f - a function that returns an image as a tensor
    params, image_f = param_f()

    if optimizer is None:
        optimizer = lambda params: torch.optim.Adam(params, lr=5e-2)
    optimizer = optimizer(params)

    if transforms is None:
        transforms = transform.standard_transforms
    transforms = transforms.copy()

    if preprocess:
        if model._get_name() == "InceptionV1":
            # Original Tensorflow InceptionV1 takes input range [-117, 138]
            transforms.append(transform.preprocess_inceptionv1())
        else:
            # Assume we use normalization for torchvision.models
            # See https://pytorch.org/docs/stable/torchvision/models.html
            transforms.append(transform.normalize())

    # Upsample images smaller than 224
    image_shape = image_f().shape
    if fixed_image_size is not None:
        new_size = fixed_image_size
    elif image_shape[2] < 224 or image_shape[3] < 224:
        new_size = 224
    else:
        new_size = None
    if new_size:
        transforms.append(
            torch.nn.Upsample(size=new_size, mode="bilinear", align_corners=True)
        )

    transform_f = transform.compose(transforms)

    hook, features = hook_model(model, image_f, return_hooks=True)
    objective_f = objectives.as_objective(objective_f)

    if verbose:
        model(transform_f(image_f()))
        print("Initial loss: {:.3f}".format(objective_f(hook)))

    images = []
    try:
        for i in tqdm(range(1, max(thresholds) + 1), disable=(not progress)):
            def closure():
                optimizer.zero_grad()
                try:
                    model(transform_f(image_f()))
                except RuntimeError as ex:
                    if i == 1:
                        # Only display the warning message
                        # on the first iteration, no need to do that
                        # every iteration
                        warnings.warn(
                            "Some layers could not be computed because the size of the "
                            "image is not big enough. It is fine, as long as the non"
                            "computed layers are not used in the objective function"
                            f"(exception details: '{ex}')"
                        )
                loss = objective_f(hook)
                loss.backward()
                return loss
                
            optimizer.step(closure)
            if i in thresholds:
                image = tensor_to_img_array(image_f())
                if verbose:
                    print("Loss at step {}: {:.3f}".format(i, objective_f(hook)))
                    if show_inline:
                        show(image)
                images.append(image)
    except KeyboardInterrupt:
        print("Interrupted optimization at step {:d}.".format(i))
        if verbose:
            print("Loss at step {}: {:.3f}".format(i, objective_f(hook)))
        images.append(tensor_to_img_array(image_f()))

    # Clear hooks
    for module_hook in features.values():
        del module_hook.module._forward_hooks[module_hook.hook.id]

    if save_image:
        export(image_f(), image_name)
    if show_inline:
        show(tensor_to_img_array(image_f()))
    elif show_image:
        view(image_f())
    return images


def tensor_to_img_array(tensor):
    image = tensor.cpu().detach().numpy()
    image = np.transpose(image, [0, 2, 3, 1])
    # Check if the image is single channel and convert to 3-channel
    if len(image.shape) == 4 and image.shape[3] == 1:  # Single channel image
        image = np.repeat(image, 3, axis=3)
    return image



In [ ]:
# pure
thresholds = (512,)
svizs = render_vis(
    model, 
    exact_tensor_all_chans(layer_name, torch.tensor(all_patches[0][None])), 
    verbose=True, 
    show_image=False, 
    thresholds=thresholds
)
plt.imshow(svizs[-1][0])
plt.show()

In [ ]:
# pure
thresholds = (512,)
svizs = render_vis(
    model, 
    exact_tensor_all_chans(layer_name, torch.tensor(all_patches[1][None])), 
    verbose=True, 
    show_image=False, 
    thresholds=thresholds
)
plt.imshow(svizs[-1][0])
plt.show()

In [ ]:
from olt.tfms import transform
timg = transform(Image.fromarray(np.floor(svizs[-1][0] * 255).astype(np.uint8)))[None]
torch.argmax(model(timg))

In [ ]:
# well it does confuse the model enough to give us a cat label lol
# it would be nice to see the activation our neuron gets. i dont think we'll be able to find anything
imagenet_label

In [ ]:
acts = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4e_1x1_pre_relu_conv"])

In [ ]:
pws = []
for r in range(3,8):
    for c in range(3,8):
        a = acts["mixed4e_1x1_pre_relu_conv"]["input"][0, :, r,c]
        layer_weight = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].detach().reshape(-1)
        pw = a * layer_weight
        pws.append(pw)


In [ ]:
# i dont know if there is something i recognise here lol. 
S([pw.reshape(22,24) for pw in pws], 25, 5)

In [ ]:
all_pws.mean(axis=0).shape

In [ ]:
from sklearn.decomposition import PCA
centered = normalize(all_pws - all_pws.mean(axis=0), "l2")
pca = PCA(100).fit(centered)
plt.plot(pca.explained_variance_)


In [ ]:
pca = PCA(40).fit(centered)
tfmed = pca.transform(centered)

In [ ]:
cluster = HDBSCAN(min_cluster_size=5, cluster_selection_method="leaf")
cluster.fit(tfmed)

In [ ]:
np.unique(cluster.labels_, return_counts=True)

In [ ]:
idxs = np.argwhere(cluster.labels_ == 3)
hms = make_heatmaps([timgs[i.item()] for i in idxs[:10]], imagenet_label)
show_grid_matplotlib(hms, 1, 10)

In [ ]:
idxs = np.argwhere(cluster.labels_ == 4)
hms = make_heatmaps([timgs[i.item()] for i in idxs[:20]], imagenet_label)
show_grid_matplotlib(hms, 2, 10)

In [ ]:
dl = MiniBatchDictionaryLearning(50)
dl.fit(all_pws)

In [ ]:
codes = dl.transform(all_pws)

In [ ]:
plt.plot(codes[0])